## Helper functions and packages

In [ ]:
# functions

from ensemble_chaos_tools import EnsembleChaos
from ensemble_chaos_tools import fix_lat_lon
from ensemble_chaos_tools import check_chaos
import xarray as xr
import glob
import matplotlib.pyplot as plt
import numpy as np
import nicopal as ncp
import cartopy.crs as ccrs

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format="jpeg"

In [ ]:
france_latitude_bnd = slice(42, 51)
france_longitude_bnd = slice(-5, 8)
paris_latitude = 49
paris_longitude = 2.5

bc_latitude_bnd = slice(48, 60)
bc_longitude_bnd = slice(-139, -114)
lytton_latitude = 50.23
lytton_longitude = -121.58

northern_hemisphere_latitude_bnd = slice(0, 90)
northern_hemisphere_longitude_bnd = slice(-180, 180)

france_centered_latitude_bnd=slice(15,70)
france_centered_longitude_bnd=slice(-60,60)

## With my AIFS ensembles

# Tests

helper

### Test of seed and deterministic optim

In [ ]:
file_path_reprod_tests_2t_seed = "/homedata/pchevali/AIFS_OUTPUTS_REGRIDDED/2025062500-tests_reproducibility_seed-uniform-0e+00-q/aifs_ensemble-2025062500-tests_reproducibility-uniform-0e+00-q-2t.nc"
file_path_reprod_tests_2t_seed_deterministic = "/homedata/pchevali/AIFS_OUTPUTS_REGRIDDED/2025062500-tests_reproducibility_seed_+_deterministic-uniform-0e+00-q/aifs_ensemble-2025062500-tests_reproducibility-uniform-0e+00-q-2t.nc"

reprod_tests_2t_seed = xr.open_dataset(
    file_path_reprod_tests_2t_seed, decode_timedelta=True
)
reprod_tests_2t_seed_deterministic = xr.open_dataset(
    file_path_reprod_tests_2t_seed_deterministic, decode_timedelta=True
)

In [ ]:
check_chaos(
    (
        reprod_tests_2t_seed.sel(
            latitude=paris_latitude, longitude=paris_longitude, method="nearest"
        )["2t"]
        - 273.15
    ),
    "Seed",
    time_of_day=[12],
    save="Seed"
)
check_chaos(
    (
        reprod_tests_2t_seed_deterministic.sel(
            latitude=paris_latitude, longitude=paris_longitude, method="nearest"
        )["2t"]
        - 273.15
    ),
    "Seed + Deterministic algorithms",
    time_of_day=[12],
    save="Seed + Deterministic algorithms"
)

### Ensemble simulations

In [ ]:
dates = ["2021061400", "2025062500"]
variables = ["2t", "z_500", "10u", "10v"]

perturbation_configs = {
    "uniform": {
        "slice": slice(1, 6),
        "exp_name": "test_perturbation_deterministic-uniform",
    },
    "gaussian": {
        "slice": slice(1, 6),
        "exp_name": "test_perturbation_deterministic-gaussian",
    },
    "brownian": {
        "slice": slice(1, 10),
        "exp_name": "test_perturbation_deterministic-brownian",
    },
    "EDA": {"slice": slice(1, 9), "exp_name": "test_perturbation_deterministic-EDA"},
}

base_dir = "/homedata/pchevali/AIFS_OUTPUTS_REGRIDDED"

datasets_results = {}
chaos_results = {}

for date in dates:
    datasets_results[date] = {var: {} for var in variables}
    chaos_results[date] = {var: {} for var in variables}

    for var in variables:
        for pert_type, config in perturbation_configs.items():
            pattern = f"{base_dir}/{date}-{config['exp_name']}*/*-{var}.nc"
            file_paths = glob.glob(pattern)

            for path in file_paths:
                if pert_type in ["EDA", "CRPS"]:
                    dict_key = pert_type
                else:
                    dict_key = f"{pert_type}-" + path.split(f"{pert_type}-")[
                        -1
                    ].replace(".nc", "")

                ds = fix_lat_lon(xr.open_dataset(path, decode_timedelta=True))

                chaos = EnsembleChaos(
                    control=ds.sel(number=0),
                    perturbed=ds.sel(number=config["slice"]),
                    var_name=var,
                )

                datasets_results[date][var][dict_key] = ds
                chaos_results[date][var][dict_key] = chaos

In [ ]:
era5_202106_07 = xr.open_mfdataset(
    [
        "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/2021/t2m.202106.as1e5.GLOBAL_025.nc",
        "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/2021/t2m.202107.as1e5.GLOBAL_025.nc",
    ]
)
era5_202106_07 = era5_202106_07.sel(
    time=era5_202106_07.time.dt.hour.isin([0, 6, 12, 18])
).compute()
mask = (
    era5_202106_07.time >= np.datetime64("2021-06-14T00:00:00.000000000")
).values * (
    era5_202106_07.time <= np.datetime64("2021-07-14T00:00:00.000000000")
).values
era5_202106_07 = fix_lat_lon(era5_202106_07.isel(time=mask))

In [ ]:
era5_202506_07 = xr.open_mfdataset(
    [
        "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/2025/t2m.202506.as1e5.GLOBAL_025.nc",
        "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/2025/t2m.202507.as1e5.GLOBAL_025.nc",
    ]
)
era5_202506_07 = era5_202506_07.sel(
    time=era5_202506_07.time.dt.hour.isin([0, 6, 12, 18])
).compute()
mask = (
    era5_202506_07.time >= np.datetime64("2025-06-25T00:00:00.000000000")
).values * (
    era5_202506_07.time <= np.datetime64("2025-07-25T00:00:00.000000000")
).values
era5_202506_07 = fix_lat_lon(era5_202506_07.isel(time=mask))

In [ ]:
for k, v in datasets_results["2021061400"]["2t"].items():
    check_chaos(
        v.sel(latitude=paris_latitude, longitude=paris_longitude, method="nearest")[
            "2t"
        ]
        - 273.15,
        title=k,
        era=era5_202106_07.sel(
            latitude=paris_latitude, longitude=paris_longitude, method="nearest"
        )["t2m"]-273.15,
        time_of_day=[12],
        save=k
    )
for k, v in datasets_results["2025062500"]["2t"].items():
    check_chaos(
        v.sel(latitude=paris_latitude, longitude=paris_longitude, method="nearest")[
            "2t"
        ]
        - 273.15,
        title=k,
        era=era5_202506_07.sel(
            latitude=paris_latitude, longitude=paris_longitude, method="nearest"
        )["t2m"]-273.15,
        time_of_day=[12],
        save=k
    )

In [ ]:
for k, v in chaos_results["2021061400"]['z_500'].items():
    print(k)
    v.growth_rate_pairwise_bootstrap(
        lat=france_centered_latitude_bnd,
        lon=france_centered_longitude_bnd,
        times=[12],
        save=k,
    )
for k, v in chaos_results["2025062500"]["z_500"].items():
    print(k)
    v.growth_rate_pairwise_bootstrap(
        lat=france_centered_latitude_bnd,
        lon=france_centered_longitude_bnd,
        times=[12],
        save=k,
    )

example of slow divergence

In [ ]:
slow_dataarray=datasets_results['2025062500']['2t']["uniform-1e-03-q-2t"]
check_chaos(
    slow_dataarray.sel(latitude=paris_latitude, longitude=paris_longitude, method="nearest")[
        "2t"
    ]
    - 273.15,
    title="uniform-1e-03-q-2t",
    time_of_day=[12],
    save="uniform-1e-03-q-2t"
)

examples of damping

In [ ]:
damping_dataarray=datasets_results['2025062500']['2t']['uniform-1e-02-2t-2t']
damping_chaos_obj=chaos_results['2025062500']['2t']['uniform-1e-02-2t-2t']
check_chaos(
    damping_dataarray.sel(latitude=paris_latitude, longitude=paris_longitude, method="nearest")[
        "2t"
    ]
    - 273.15,
    title="uniform-1e-02-2t-2t",
    time_of_day=[0],
    save="uniform-1e-02-2t-2t"
)
damping_chaos_obj.growth_rate_pairwise_bootstrap(
        lat=northern_hemisphere_latitude_bnd,
        lon=northern_hemisphere_longitude_bnd,
        times=[0,6,12,18],
        save='uniform-1e-02-2t-2t'
    )

### Fields vs ERA5

In [ ]:
vmax_pred = (
    chaos_results["2025062500"]["2t"]["uniform-1e-02-2t-2t"].perturbed.sel(number=4)["2t"].max().item()
)
vmin_pred = (
    chaos_results["2025062500"]["2t"]["uniform-1e-02-2t-2t"].perturbed.sel(number=4)["2t"].min().item()
)

vmax_era5 = era5_202506_07["t2m"].max().item()
vmin_era5 = era5_202506_07["t2m"].min().item()

vmax = max(vmax_pred, vmax_era5)
vmin = min(vmin_pred, vmin_era5)

aifs_valid_times = (
    chaos_results["2025062500"]["2t"]["uniform-1e-02-2t-2t"].perturbed.sel(number=4).valid_time.values
)
era5_times = era5_202506_07.time.values

for i in range(5):
    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(12, 4), subplot_kw={"projection": ccrs.PlateCarree()}
    )

    chaos_results["2025062500"]["2t"]["uniform-1e-02-2t-2t"].perturbed.sel(number=4).isel(step=i)[
        "2t"
    ].plot.contourf(ax=ax1, vmin=vmin, vmax=vmax)
    ax1.coastlines()
    ax1.set_title(f"AIFS ({aifs_valid_times[i]})")

    era5_202506_07.isel(time=i)["t2m"].plot.contourf(ax=ax2, vmin=vmin, vmax=vmax)
    ax2.coastlines()
    ax2.set_title(f"ERA5 (Time {era5_times[i]})")

    plt.show()

In [ ]:
vmax_pred = (
    chaos_results["2025062500"]["2t"]["brownian-1e-01-u-v-2t"]
    .perturbed.sel(number=4)["2t"]
    .max()
    .item()
)
vmin_pred = (
    chaos_results["2025062500"]["2t"]["brownian-1e-01-u-v-2t"]
    .perturbed.sel(number=4)["2t"]
    .min()
    .item()
)

vmax_era5 = era5_202506_07["t2m"].max().item()
vmin_era5 = era5_202506_07["t2m"].min().item()

vmax = max(vmax_pred, vmax_era5)
vmin = min(vmin_pred, vmin_era5)

aifs_valid_times = (
    chaos_results["2025062500"]["2t"]["brownian-1e-01-u-v-2t"]
    .perturbed.sel(number=4)
    .valid_time.values
)
era5_times = era5_202506_07.time.values

for i in range(5):
    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(12, 4), subplot_kw={"projection": ccrs.PlateCarree()}
    )

    chaos_results["2025062500"]["2t"]["brownian-1e-01-u-v-2t"].perturbed.sel(
        number=4
    ).isel(step=i)["2t"].plot.contourf(ax=ax1, vmin=vmin, vmax=vmax)
    ax1.coastlines()
    ax1.set_title(f"AIFS ({aifs_valid_times[i]})")

    era5_202506_07.isel(time=i)["t2m"].plot.contourf(ax=ax2, vmin=vmin, vmax=vmax)
    ax2.coastlines()
    ax2.set_title(f"ERA5 (Time {era5_times[i]})")

    plt.show()

In [ ]:
vmax_pred = (
    chaos_results["2025062500"]["2t"]["brownian-1e-01-2t-2t"]
    .perturbed.sel(number=4)["2t"]
    .max()
    .item()
)
vmin_pred = (
    chaos_results["2025062500"]["2t"]["brownian-1e-01-2t-2t"]
    .perturbed.sel(number=4)["2t"]
    .min()
    .item()
)

vmax_era5 = era5_202506_07["t2m"].max().item()
vmin_era5 = era5_202506_07["t2m"].min().item()

vmax = max(vmax_pred, vmax_era5)
vmin = min(vmin_pred, vmin_era5)

aifs_valid_times = (
    chaos_results["2025062500"]["2t"]["brownian-1e-01-2t-2t"]
    .perturbed.sel(number=4)
    .valid_time.values
)
era5_times = era5_202506_07.time.values

for i in range(5):
    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(12, 4), subplot_kw={"projection": ccrs.PlateCarree()}
    )

    chaos_results["2025062500"]["2t"]["brownian-1e-01-2t-2t"].perturbed.sel(
        number=4
    ).isel(step=i)["2t"].plot.contourf(ax=ax1, vmin=vmin, vmax=vmax)
    ax1.coastlines()
    ax1.set_title(f"AIFS ({aifs_valid_times[i]})")

    era5_202506_07.isel(time=i)["t2m"].plot.contourf(ax=ax2, vmin=vmin, vmax=vmax)
    ax2.coastlines()
    ax2.set_title(f"ERA5 (Time {era5_times[i]})")

    plt.show()

### Maps as gif

In [ ]:
chaos_results["2025062500"]["z_500"]["uniform-1e-02-2t-z_500"].plot_nice_looking_animation(
    slice(-80, 80),
    slice(-170, 170),
    filename="SINGLE_uniform-1e-02-2t_z_500_0.gif",
    cmap=ncp.pal("Lithium"),
    speed=200,
)
chaos_results["2025062500"]["z_500"]["uniform-1e-02-2t-z_500"].plot_nice_looking_animation(
    slice(-80, 80),
    slice(-170, 170),
    filename="SINGLE_uniform-1e-02-2t_z_500_4.gif",
    cmap=ncp.pal("Lithium"),
    speed=200,
    member=4,
)
chaos_results["2025062500"]["z_500"]["uniform-1e-02-2t-z_500"].plot_nice_looking_animation(
    france_centered_latitude_bnd,
    france_centered_longitude_bnd,
    filename="SINGLE_uniform-1e-02-2t_z_500_0_zoom.gif",
    cmap=ncp.pal("Lithium"),
    speed=200,
)
chaos_results["2025062500"]["z_500"]["uniform-1e-02-2t-z_500"].plot_nice_looking_animation(
    france_centered_latitude_bnd,
    france_centered_longitude_bnd,
    filename="SINGLE_uniform-1e-02-2t_z_500_4_zoom.gif",
    cmap=ncp.pal("Lithium"),
    speed=200,
    member=4,
)

## Tests spatially correlated noise

In [ ]:
for i in range(10):
    field = (
        datasets_results["2025062500"]["2t"]["EDA"].sel(number=0).isel(step=0)["2t"]
        - datasets_results["2025062500"]["2t"]["EDA"].sel(number=i).isel(step=0)["2t"]
    )
    field.plot()
    plt.show()

In [ ]:
def _generate_brownian_noise(shape, reddening=2, scale=0.5):
    """Utility to produce brownian noise using FFT (Bourke 1997)"""
    noise = np.random.normal(loc=0.0, scale=scale, size=shape)
    x_white = np.fft.rfft2(noise)

    S = (np.abs(np.fft.fftfreq(shape[-2]).reshape(-1, 1)) ** reddening) + (
        np.fft.rfftfreq(shape[-1]) ** reddening
    )

    with np.errstate(divide="ignore", invalid="ignore"):
        S = 1.0 / S
    S[..., 0, 0] = 0.0
    S = S / np.sqrt(np.mean(S**2))

    x_shaped = x_white * S
    noise_shaped = np.fft.irfft2(x_shaped, s=(shape[-2], shape[-1]))

    return noise_shaped

In [ ]:
plt.imshow(
    _generate_brownian_noise((721, 1440), reddening=2, scale=0.33),
    cmap="Greys",
)
plt.colorbar(shrink=0.5)
plt.grid(None)
plt.xticks([])
plt.yticks([])
plt.savefig("example_fBm.pdf",bbox_inches="tight")
plt.show()